# Structured Output: Practice Exercise

Build a structured output extractor that analyzes book reviews and returns consistent, typed data. You will use the LangChain/Pydantic approach from the lesson.

**What you'll implement:**
- A Pydantic model defining the structure for book review analysis
- A structured output model that extracts review data from natural language

**Estimated time:** 10 minutes

## Setup

Run this cell to import the required libraries and initialize the API client.

In [ ]:
%pip install -qU langchain-openai
%pip install -qU langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.1 MB/s eta 0:00:00


In [ ]:
# Setup - run this cell first

import os
import json
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Optional, List
from dotenv import load_dotenv

from google.colab import userdata

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    print("OpenAI API key found. Ready to proceed!")
else:
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')


# Initialize base model for later use
base_model = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPEN_AI_API_KEY)

## Context

You are building a book review analysis system. Given free-form book review text, you need to extract structured information that can be stored in a database or used by downstream systems.

**Input:** Natural language book reviews like:
- "I just finished reading 'The Great Gatsby' by F. Scott Fitzgerald. Gave it 4 out of 5 stars. The prose is beautiful but the pacing felt slow at times."
- "5 stars for Dune! Frank Herbert created an incredible universe. Highly recommend for sci-fi fans."

**Expected output structure:**
- `book_title` (string): The title of the book
- `author` (string or null): The author's name, if mentioned
- `rating` (integer): Rating from 1-5
- `recommendation` (boolean): Whether the reviewer recommends the book

Your Pydantic model should handle cases where the author is not mentioned (use `Optional` with a default of `None`).

## Part 1: Define the Pydantic Model

Create a Pydantic model class that defines the structure for book review analysis.

In [ ]:
class BookReviewAnalysis(BaseModel):
    """Model for extracting structured information from book reviews.

    Extracts the book title, author (if mentioned), numeric rating,
    and whether the reviewer recommends the book.
    """

    # TODO: Define the four fields with appropriate types and Field descriptions:
    # - book_title: required string
    # - author: optional string (can be None if not mentioned)
    # - rating: required integer between 1-5
    # - recommendation: required boolean
    book_title:str = Field(description="Title of the book")
    author:Optional[str] = Field("Author of the book")
    rating:int = Field(le=5,ge=1,description="Rating for the movie")
    recommendation:bool =Field(description="Recommendation")


## Part 2: Create Structured Output Model and Extract Data

Use LangChain's `with_structured_output()` method to create a model that returns `BookReviewAnalysis` objects.

In [ ]:
def analyze_book_review(review_text: str) -> BookReviewAnalysis:
    """
    Analyze a book review and extract structured information.

    Args:
        review_text: Natural language book review text

    Returns:
        BookReviewAnalysis object with extracted fields
    """
    # TODO: Create a structured output model using base_model.with_structured_output()
    # and invoke it with the review_text
    structured_output_model = base_model.with_structured_output(BookReviewAnalysis)
    return structured_output_model.invoke(review_text)

## Run Your Implementation

Test your implementation with these sample reviews.

In [ ]:
# Test reviews
test_reviews = [
    "I just finished reading 'The Great Gatsby' by F. Scott Fitzgerald. Gave it 4 out of 5 stars. The prose is beautiful but the pacing felt slow at times.",
    "5 stars for Dune! Frank Herbert created an incredible universe. Highly recommend for sci-fi fans.",
    "Read '1984' last week. Gave it 3 stars - important themes but quite depressing. Not sure I'd recommend it to everyone."
]

for i, review in enumerate(test_reviews, 1):
    print(f"\n{'='*60}")
    print(f"Review {i}")
    print(f"{'='*60}")
    print(f"Input: {review[:80]}...")

    result = analyze_book_review(review)

    print(f"\nExtracted data:")
    print(json.dumps(result.model_dump(), indent=2))


Review 1
Input: I just finished reading 'The Great Gatsby' by F. Scott Fitzgerald. Gave it 4 out...

Extracted data:
{
  "book_title": "The Great Gatsby",
  "author": "F. Scott Fitzgerald",
  "rating": 4,
  "recommendation": true
}

Review 2
Input: 5 stars for Dune! Frank Herbert created an incredible universe. Highly recommend...

Extracted data:
{
  "book_title": "Dune",
  "author": "Frank Herbert",
  "rating": 5,
  "recommendation": true
}

Review 3
Input: Read '1984' last week. Gave it 3 stars - important themes but quite depressing. ...

Extracted data:
{
  "book_title": "1984",
  "author": null,
  "rating": 3,
  "recommendation": false
}
